<img alt="Logo da Universidade Federal do Ceará" src="../assets/logo-ufc.svg">

# **Análise Exploratória dos Dados**

### **Objetivo do Notebook**

Este notebook realiza a análise exploratória dos dados preparados em [`preprocessing.ipynb`](preprocessing.ipynb). Ele lê os arquivos de saída e apresenta estatísticas descritivas da distância até as unidades de saúde, em linha reta e pela malha viária, e da parcela da população desassistida por faixa de renda. Os resultados orientam e justificam a escolha e a verificação dos limiares adotados no projeto.

**Fluxo de Trabalho**

| Etapa | O que acontece |
|---|---|
| 0. Configuração | Leitura dos arquivos pré-processados e funções auxiliares |
| 1. Distâncias | Percentis de distância ponderados por pessoas, em linha reta e pela via |
| 2. Limiares | Limiares adotados e justificativa |
| 3. Desassistência por renda | Parcela da população além de cada limiar, por faixa de renda |
| 4. Efeito do limiar rígido | Comparação entre limiar único e limiar rígido em zonas vulneráveis |
| 5. Hipóteses | Primeira leitura dos resultados frente às hipóteses da proposta |
| 6. Limitações | Limitações da análise |


## **0. Configuração**

Seção destinada à leitura dos arquivos pré-processados e à definição das funções auxiliares. Os limiares e a regra do limiar rígido são lidos de `metadados.json`, de modo que o notebook sempre use os mesmos parâmetros do pré-processamento.


In [7]:
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
pd.set_option("display.max_columns", 30)

DIR_SAIDA = Path("../data/output")   # Arquivos gerados pelo pré-processamento

setores = pd.read_csv(DIR_SAIDA / "setores.csv", dtype={"CD_SETOR": str, "CD_BAIRRO": str})
with open(DIR_SAIDA / "metadados.json", encoding="utf-8") as f:
    meta = json.load(f)

unidades = pd.read_csv(DIR_SAIDA / "unidades.csv")
bairros = pd.DataFrame(gpd.read_file(DIR_SAIDA / "bairros.geojson").drop(columns="geometry"))

LIMIARES = {nivel: tuple(v) for nivel, v in meta["limiares_m"].items()}
NIVEIS = {"UBS": "ubs", "UPA": "upa", "Hospital": "hosp"}   # Nível -> sufixo das colunas
ORDEM_FAIXAS = ["Todas", *meta["renda"]["faixas"], "Sem dado"]

# Carga da unidade mais próxima (em linha reta) de cada setor
for k in NIVEIS.values():
    setores[f"carga_{k}"] = setores[f"un_{k}"].map(unidades.set_index("id")["carga"])

print(f"Setores: {len(setores):,} | Bairros: {len(bairros):,} | Unidades: {len(unidades):,} | "
      f"População: {setores['populacao'].sum():,.0f}")
print(f"Limiares (m): {LIMIARES}")
print(f"Limiar rígido: x{meta['limiar_rigido']['fator']} em setores com renda média até "
      f"{meta['limiar_rigido']['renda_sm_ate']} SM")

Setores: 4,268 | Bairros: 121 | Unidades: 193 | População: 2,412,350
Limiares (m): {'UBS': (1000, 1300), 'UPA': (3500, 4500), 'Hospital': (3000, 4000)}
Limiar rígido: x0.8 em setores com renda média até 1 SM


In [18]:
def quantil_ponderado(valores, pesos, qs):
    """Calcula quantis ponderados de uma lista de valores, com pesos. Retorna uma lista de quantis na mesma ordem de qs."""
    # Menor valor cuja população acumulada alcança cada quantil em qs
    d = pd.DataFrame({"v": valores, "p": pesos}).dropna().sort_values("v")
    v = d["v"].to_numpy()
    acum = (d["p"].cumsum() / d["p"].sum()).to_numpy()
    return [float(v[min(np.searchsorted(acum, q), len(v) - 1)]) for q in qs]


def gini(valores, pesos=None):
    """Índice de Gini (0 = igualdade; 1 = desigualdade máxima), com pesos opcionais."""
    v = np.asarray(valores, float)
    w = np.ones_like(v) if pesos is None else np.asarray(pesos, float)
    o = np.argsort(v)
    v, w = v[o], w[o]
    f, l = np.cumsum(w) / w.sum(), np.cumsum(v * w) / (v * w).sum()
    return float(1 - np.sum((f - np.r_[0, f[:-1]]) * (l + np.r_[0, l[:-1]])))


def pct_pop(df, mascara):
    """Porcentagem da população de df que atende à máscara (série booleana alinhada a df)."""
    return 100 * df.loc[mascara, "populacao"].sum() / df["populacao"].sum()

## **1. Distribuição das distâncias**

Percentis da distância até a unidade mais próxima, ponderados por população, em linha reta e pela malha viária (OSM). A segunda tabela mostra quanto o percurso pela via é maior que a linha reta, pela mediana, ponderada por população, da razão entre as duas distâncias em cada setor.


In [19]:
# Percentis de distância (m), ponderados por população
qs = [0.5, 0.75, 0.9, 0.95]


def tabela_percentis(prefixo):
    return pd.DataFrame(
        {nivel: quantil_ponderado(setores[f"{prefixo}_{k}"], setores["populacao"], qs) for nivel, k in NIVEIS.items()},
        index=[f"p{int(q * 100)}" for q in qs],
    )


tabela_dist = pd.concat({"Linha reta": tabela_percentis("dist"), "Via (OSM)": tabela_percentis("dist_via")}, axis=1)
tabela_dist

Linha reta                  Via (OSM)                 
           UBS     UPA Hospital       UBS     UPA Hospital
p50      541.0 2,371.0  1,426.0     814.0 3,250.0  2,046.0
p75      790.0 3,530.0  2,125.0   1,143.0 4,848.0  2,919.0
p90    1,075.0 4,466.0  3,093.0   1,527.0 5,988.0  4,145.0
p95    1,290.0 5,158.0  3,837.0   1,822.0 6,681.0  5,006.0

In [20]:
# Razão via / linha reta (setores com distância em linha reta nula são ignorados)
razao = {
    nivel: quantil_ponderado(setores[f"dist_via_{k}"] / setores[f"dist_{k}"].replace(0, np.nan), setores["populacao"], [0.5])[0]
    for nivel, k in NIVEIS.items()
}
pd.Series(razao, name="mediana via / linha reta").to_frame().T.style.format("{:.2f}")

,UBS,UPA,Hospital
mediana via / linha reta,1.46,1.36,1.40


## **2. Limiares de desassistência**

### **Limiares adotados e justificativa**

| Serviço | Moderado | Severo | Referência empírica (linha reta) |
|---|---|---|---|
| UBS | 1000 m | 1300 m | p90 (1075 m) e p95 (1290 m) |
| UPA | 3500 m | 4500 m | p75 (3530 m) e p90 (4466 m) |
| Hospital | 3000 m | 4000 m | p90 (3093 m) e p95 (3837 m) |

A escolha combinou critérios normativos e empíricos:

1. **Critério normativo:** parte da distância de percurso que cada tipo de serviço comporta.
    * **UBS (acesso a pé):** equipamentos de uso diário ou preventivo devem estar a cerca de 800 m a 1 km de caminhada. Acima de 1 km a 1,5 km, a necessidade de pagar passagem ou depender de carona para um serviço primário favorece o abandono de tratamentos contínuos, como pré-natal e hipertensão.
    * **UPA (acesso motorizado):** por ser um serviço de urgência, a referência é uma distância de cerca de 3 km a 5 km pela malha viária. Acima disso, a distância passa a configurar forte barreira de acesso.
    * **Hospital (alta complexidade):** os hospitais operam por distritos ou microrregiões de saúde, e o acesso não precisa ser local. Entretanto, os limiares adotados são mais exigentes do que a referência, que é cerca de 5 km a 10 km pela malha viária.

2. **Critério empírico:** o limiar moderado foi ancorado entre os percentis p75 e p90 e o severo entre p90 e p95 da distribuição de distâncias ponderada por pessoas. Isso garante prevalência suficiente para comparar faixas de renda e, ao mesmo tempo, isola a cauda de maior desassistência. Esse critério, porém, tem um ponto cego: definir o que é adequado a partir da própria distribuição do território tende a normalizar o déficit de infraestrutura. Se a cidade tem poucas unidades, a distribuição será ruim, e calibrar o modelo por ela equivale a aceitar a desassistência como padrão. Por isso, o critério empírico é usado como complementar ao normativo e não define sozinho o que é adequado.

3. **Natureza conservadora da distância em linha reta:** o percurso real é maior que a linha reta por causa da malha viária (neste projeto, a mediana da razão entre os dois é da ordem de 1,4). Os resultados em linha reta devem, portanto, ser lidos como um limite inferior da desassistência. Os mesmos limiares são aplicados à distância pela via, pois os critérios normativos se referem a percursos.

4. **Limiar mais rígido em zonas de vulnerabilidade econômica:** a mesma distância pesa mais para quem depende de deslocamento a pé ou de transporte público. Nos setores com renda média de até 1 SM, os limiares são multiplicados por um fator (0,8), o que os torna mais exigentes.

Por fim, a classificação em três classes (adequado, moderada e severa) tem o objetivo de mostrar a desassistência sem esconder os dados reais.

## **3. Desassistência por faixa de renda**

Visualização e análise da parcela da população de cada faixa de renda além do limiar moderado (desassistência moderada ou severa) e além do limiar severo, em cada nível. Em cada tabela, as colunas são normalizadas separadamente no mapa de cores, para que limiares com escalas muito diferentes (por exemplo, 0,0% e 60%) continuem comparáveis entre faixas.

In [21]:
def tabela_desassistencia(prefixo):
    """Porcentagem da população por faixa de renda além do limiar moderado e do severo. prefixo: 'cls' (linha reta) ou 'cls_via'."""
    linhas = []
    for faixa, g in [("Todas", setores), *setores.groupby("faixa_renda")]:
        linha = {"faixa": faixa}
        for nivel, k in NIVEIS.items():
            classe = g[f"{prefixo}_{k}"]
            linha[f"{nivel} > moderado (%)"] = pct_pop(g, classe >= 1)
            linha[f"{nivel} > severo (%)"] = pct_pop(g, classe == 2)
        linhas.append(linha)
    t = pd.DataFrame(linhas).set_index("faixa")
    return t.reindex([f for f in ORDEM_FAIXAS if f in t.index])


def estilo(t, titulo):
    return t.style.background_gradient(cmap="YlOrRd", axis=0).format("{:.1f}").set_caption(titulo)


estilo(tabela_desassistencia("cls"), "Linha reta: % da população além de cada limiar, por faixa de renda")

,UBS > moderado (%),UBS > severo (%),UPA > moderado (%),UPA > severo (%),Hospital > moderado (%),Hospital > severo (%)
faixa,,,,,,
Todas,14.1,5.2,26.7,11.0,12.7,5.3
Até 2 SM,7.6,1.5,19.5,5.7,18.1,7.8
De 2 a 5 SM,16.5,4.2,31.0,13.5,2.9,0.8
Mais de 5 SM,47.1,28.5,60.0,36.7,0.1,0.1
Sem dado,12.6,0.0,20.2,2.7,12.6,4.4


In [22]:
estilo(tabela_desassistencia("cls_via"), "Via (OSM): % da população além de cada limiar, por faixa de renda")

,UBS > moderado (%),UBS > severo (%),UPA > moderado (%),UPA > severo (%),Hospital > moderado (%),Hospital > severo (%)
faixa,,,,,,
Todas,36.6,18.6,47.1,31.0,25.5,12.9
Até 2 SM,29.1,12.0,41.0,25.0,33.8,18.2
De 2 a 5 SM,41.0,21.3,51.9,32.7,10.9,3.9
Mais de 5 SM,71.5,52.1,73.2,62.8,6.8,0.3
Sem dado,41.5,17.5,45.4,27.3,32.2,15.3


## **4. Análise do efeito do limiar rígido**

Nessa seção, é feita a comparação entre o limiar único (o mesmo para todos os setores) e o limiar rígido (multiplicado pelo fator nos setores vulneráveis). A primeira tabela considera somente a população de setores vulneráveis, em que a regra se aplica; a segunda, toda a população.

In [28]:
def efeito_rigor(df):
    linhas = []
    for base, p_dist, p_cls in [("Linha reta", "dist", "cls"), ("Via (OSM)", "dist_via", "cls_via")]:
        for nivel, k in NIVEIS.items():
            moderado, severo = LIMIARES[nivel]
            d, classe = df[f"{p_dist}_{k}"], df[f"{p_cls}_{k}"]
            linhas.append({
                "base": base, "nível": nivel,
                "moderada, único (%)": pct_pop(df, d > moderado),
                "moderada, rígido (%)": pct_pop(df, classe >= 1),
                "severa, único (%)": pct_pop(df, d > severo),
                "severa, rígido (%)": pct_pop(df, classe == 2),
            })
    return pd.DataFrame(linhas).set_index(["base", "nível"])


print("População de setores vulneráveis")
display(efeito_rigor(setores[setores["vulneravel"]]).style.format("{:.1f}"))
print("Toda a população")
display(efeito_rigor(setores).style.format("{:.1f}"))

População de setores vulneráveis


Toda a população


## **5. Análises iniciais frente às hipóteses**

As hipóteses da proposta que podem ser examinadas nos dados pré-processados são lidas a seguir (H1, H2, H3, H4), com a distância em linha reta e análise viária. A hipótese H5, sobre o ganho de uma nova unidade, depende da simulação no site e não será examinada.

### **H1: distância à UBS curta e semelhante entre bairros**

Distância mediana de cada bairro (ponderada pela população dos setores) e sua dispersão entre os bairros.


In [29]:
# Tabela 1: Distância Padrão (Euclidiana)
linhas_padrao = []
for nivel, k in NIVEIS.items():
    d = bairros[f"dist_{k}"]
    linhas_padrao.append({
        "nível": nivel, 
        "mediana dos bairros (m)": d.median(), 
        "p10 (m)": d.quantile(0.1), 
        "p90 (m)": d.quantile(0.9),
        "p90 / p10": d.quantile(0.9) / d.quantile(0.1),
        "% da população além do limiar moderado": pct_pop(setores, setores[f"cls_{k}"] >= 1),
    })

tabela_padrao = pd.DataFrame(linhas_padrao).set_index("nível")
display(tabela_padrao.style.format("{:.1f}").set_caption("Tabela 1: Distância Linear (Padrão)"))

# Tabela 2: Distância Viária
linhas_viaria = []
for nivel, k in NIVEIS.items():
    d_via = bairros[f"dist_via_{k}"]
    linhas_viaria.append({
        "nível": nivel, 
        "mediana dos bairros (m)": d_via.median(), 
        "p10 (m)": d_via.quantile(0.1), 
        "p90 (m)": d_via.quantile(0.9),
        "p90 / p10": d_via.quantile(0.9) / d_via.quantile(0.1),
        "% da população além do limiar moderado": pct_pop(setores, setores[f"cls_via_{k}"] >= 1),
    })

tabela_viaria = pd.DataFrame(linhas_viaria).set_index("nível")
display(tabela_viaria.style.format("{:.1f}").set_caption("Tabela 2: Distância Viária"))

,mediana dos bairros (m),p10 (m),p90 (m),p90 / p10,% da população além do limiar moderado
nível,,,,,
UBS,549.0,361.0,1024.0,2.8,14.1
UPA,2629.0,1054.0,4639.0,4.4,26.7
Hospital,1418.0,651.0,2911.0,4.5,12.7


,mediana dos bairros (m),p10 (m),p90 (m),p90 / p10,% da população além do limiar moderado
nível,,,,,
UBS,809.0,584.0,1481.0,2.5,36.6
UPA,3569.0,1656.0,6182.0,3.7,47.1
Hospital,2089.0,1148.0,3991.0,3.5,25.5


### **H2: carga mais desigual que a distância**

Comparação entre o índice de Gini da carga, entre as unidades, e o da distância, entre as pessoas. Além disso, também é mostrada a parcela da população atendida pelo décimo de unidades mais carregado.


In [30]:
linhas = []
for nivel, k in NIVEIS.items():
    carga = unidades.loc[unidades["nivel"] == nivel, "carga"]
    n_top = max(1, round(0.1 * len(carga)))
    linhas.append({
        "nível": nivel, "unidades": len(carga),
        "Gini da carga": gini(carga),
        "Gini da distância": gini(setores[f"dist_{k}"], setores["populacao"]),
        "% da população nas 10% mais carregadas": 100 * carga.nlargest(n_top).sum() / carga.sum(),
    })
pd.DataFrame(linhas).set_index("nível").style.format("{:.2f}", subset=["Gini da carga", "Gini da distância"]).format("{:.1f}", subset=["% da população nas 10% mais carregadas"]).format("{:.0f}", subset=["unidades"])

,unidades,Gini da carga,Gini da distância,% da população nas 10% mais carregadas
nível,,,,
UBS,143,0.26,0.32,19.6
UPA,13,0.34,0.31,15.5
Hospital,37,0.42,0.34,29.1


### **H3: diferença entre faixas de renda, pequena na distância e maior na carga**

Distância mediana e carga média da unidade mais próxima, ponderadas pela população de cada faixa de renda.


In [26]:
linhas = {}
for faixa, g in [("Todas", setores), *setores.groupby("faixa_renda")]:
    linhas[faixa] = {}
    for nivel, k in NIVEIS.items():
        linhas[faixa][(nivel, "distância mediana (m)")] = quantil_ponderado(g[f"dist_{k}"], g["populacao"], [0.5])[0]
        linhas[faixa][(nivel, "carga média")] = np.average(g[f"carga_{k}"], weights=g["populacao"])
pd.DataFrame.from_dict(linhas, orient="index").reindex([f for f in ORDEM_FAIXAS if f in linhas]).style.format("{:,.0f}")

### **H4: o perfil de deficiência varia entre os bairros**

Para cada bairro, identifica-se a dimensão em que ele está pior em relação aos demais, isto é, aquela em que sua posição percentual entre os bairros é a mais alta (distância ou carga, em cada nível). A tabela conta quantos bairros têm cada dimensão como a mais deficiente.


In [31]:
dimensoes = {f"{nivel}: {nome}": f"{col}_{k}" for nivel, k in NIVEIS.items()
             for nome, col in [("distância", "dist"), ("carga", "carga")]}
posicao = bairros[list(dimensoes.values())].rank(pct=True)
posicao.columns = list(dimensoes)
posicao.idxmax(axis=1).value_counts().rename("bairros").to_frame()

,bairros
UPA: carga,27
Hospital: distância,24
Hospital: carga,22
UBS: carga,19
UBS: distância,15
UPA: distância,14


### **Leitura dos resultados**

* **H1 (parcialmente sustentada):** A UBS tem a menor distância mediana entre os bairros (549 m) e a menor dispersão (razão p90/p10 de 2,8, contra 4,4 da UPA e 4,5 do hospital), e a UPA concentra a maior parcela da população além do limiar moderado (26,7%). Por outro lado, a distância até o hospital não é um problema maior que a da UBS (12,7% contra 14,1% da população além do limiar moderado), e a distância até a UBS varia quase três vezes entre os bairros, de modo que não é propriamente semelhante entre eles.

* **H2 (parcialmente sustentada):** O Gini da carga supera o da distância no hospital (0,42 contra 0,34) e, por pouco, na UPA (0,34 contra 0,31), mas não na UBS (0,26 contra 0,32), nível em que a carga é mais homogênea. Ainda assim, o décimo de unidades mais carregado atende 19,6% da população na UBS, 29,1% no hospital e 15,5% na UPA (uma única unidade, em um nível com apenas 13), entre duas e três vezes a parcela proporcional.

* **H3 (não sustentada):** A distância difere bastante entre as faixas de renda: a mediana até a UBS é de 489 m na faixa "Até 2 SM" e de 945 m na faixa "Mais de 5 SM", e até a UPA, de 2.117 m e 3.911 m. Até o hospital, o padrão se inverte (1.640 m e 1.029 m). A carga média varia em proporção semelhante à da distância (na UBS, 18.735 e 33.583 pessoas), e não em proporção maior. A faixa de maior renda é a mais distante da UBS e da UPA, e a de menor renda, a mais distante do hospital. O mesmo padrão aparece na distância pela via.

* **H4 (sustentada):** Nenhuma dimensão predomina entre os 121 bairros: a mais frequente (carga da UPA) é a pior em 27 bairros e a menos frequente (distância até a UPA), em 14.